# **Etape 1 : Extraction OSM route**

### par EPCI -non

In [ ]:
# ============================================================
# INSTALLATION
# ============================================================

!pip install -q osmnx geopandas pyogrio shapely

# ============================================================
# GOOGLE DRIVE
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

# ============================================================
# PARAMETRES
# ============================================================

OUTPUT_DIR = "/content/drive/MyDrive/data_osm_road"
EPCI_FILE = "/content/drive/MyDrive/data_osm_road/basrhin1_bdtopo_3948_2026.gpkg"

# ============================================================
# IMPORTS
# ============================================================

import os
import time

import geopandas as gpd
import osmnx as ox

from shapely.validation import make_valid

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# OSMNX
# ============================================================

ox.settings.use_cache = True
ox.settings.log_console = True
ox.settings.requests_timeout = 600

# Tags supplémentaires à récupérer
ox.settings.useful_tags_way = list(
    set(
        ox.settings.useful_tags_way
        + [
            "bicycle",
            "cycleway",
            "cycleway:left",
            "cycleway:right",
            "cycleway:both",
            "surface",
            "smoothness",
            "tracktype",
            "segregated",
            "lit",
            "lanes",
            "maxspeed",
            "oneway",
            "access",
            "foot",
            "horse",
            "motor_vehicle",
            "vehicle",
            "service",
            "junction",
            "bridge",
            "tunnel",
        ]
    )
)

# ============================================================
# LECTURE EPCI
# ============================================================

epci = gpd.read_file(EPCI_FILE)

print("CRS initial :", epci.crs)
print("Nb EPCI :", len(epci))

# Conversion vers WGS84 obligatoire
epci = epci.to_crs(4326)

print("CRS après conversion :", epci.crs)

# Réparation géométries
epci["geometry"] = epci["geometry"].apply(make_valid)

# ============================================================
# COLONNES A CONSERVER
# ============================================================

KEEP_COLS = [
    "osmid",
    "highway",
    "name",
    "ref",
    "surface",
    "smoothness",
    "tracktype",
    "bicycle",
    "foot",
    "horse",
    "motor_vehicle",
    "vehicle",
    "cycleway",
    "cycleway:left",
    "cycleway:right",
    "cycleway:both",
    "oneway",
    "lanes",
    "maxspeed",
    "lit",
    "segregated",
    "access",
    "bridge",
    "tunnel",
    "junction",
    "service",
    "geometry",
]

# ============================================================
# BOUCLE EPCI
# ============================================================

for idx, row in epci.iterrows():

    try:

        nom_epci = str(row["code_siren"])

        nom_epci = (
            nom_epci.replace("/", "_")
            .replace("\\", "_")
            .replace(" ", "_")
            .replace("'", "_")
        )

        print("\n" + "=" * 70)
        print(f"{idx+1}/{len(epci)} : {nom_epci}")

        polygon = row.geometry

        if polygon.is_empty:
            print("géométrie vide")
            continue

        polygon = make_valid(polygon)

        if polygon.geom_type == "MultiPolygon":
            polygon = max(polygon.geoms, key=lambda g: g.area)

        print("Téléchargement OSM...")

        G = ox.graph_from_polygon(
            polygon,
            network_type="all",
            simplify=False,
            retain_all=True,
        )

        edges = ox.graph_to_gdfs(
            G,
            nodes=False,
            fill_edge_geometry=True,
        )

        cols = [c for c in KEEP_COLS if c in edges.columns]

        edges = edges[cols]

        outfile = os.path.join(
            OUTPUT_DIR,
            f"{nom_epci}.gpkg"
        )

        edges.to_file(
            outfile,
            layer="roads",
            driver="GPKG"
        )

        print(f"OK : {len(edges):,} routes")

        time.sleep(2)

    except Exception as e:

        print(f"ERREUR : {nom_epci}")
        print(e)

# ============================================================
# FIN
# ============================================================

print("\nTERMINÉ")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CRS initial : EPSG:3948
Nb EPCI : 10
CRS après conversion : EPSG:4326

1/10 : 200030526
Téléchargement OSM...
OK : 44,892 routes

2/10 : 200034270
Téléchargement OSM...
OK : 98,010 routes

3/10 : 200067924
Téléchargement OSM...
OK : 95,553 routes

4/10 : 246700306
Téléchargement OSM...
OK : 173,983 routes

5/10 : 246700488
Téléchargement OSM...
OK : 454,434 routes

6/10 : 246700744
Téléchargement OSM...
OK : 74,580 routes

7/10 : 246700777
Téléchargement OSM...
OK : 91,433 routes

8/10 : 246700967
Téléchargement OSM...
OK : 80,905 routes

9/10 : 246701064
Téléchargement OSM...
OK : 95,133 routes

10/10 : 246701080
Téléchargement OSM...
OK : 29,891 routes

TERMINÉ


### par département

In [ ]:
# ============================================================
# INSTALLATION
# ============================================================

!pip install -q osmnx geopandas pyogrio shapely

# ============================================================
# DRIVE
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

# ============================================================
# PARAMETRES
# ============================================================

DEP_FILE = "/content/drive/MyDrive/data_osm_road/departement_alsace.gpkg"

OUTPUT_DIR = "/content/drive/MyDrive/data_osm_road"

# ============================================================
# IMPORTS
# ============================================================

import os
import geopandas as gpd
import osmnx as ox

from shapely.validation import make_valid

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# OSMNX
# ============================================================

ox.settings.use_cache = True
ox.settings.log_console = True
ox.settings.requests_timeout = 1200

ox.settings.useful_tags_way = list(
    set(
        ox.settings.useful_tags_way + [
            "bicycle",
            "cycleway",
            "cycleway:left",
            "cycleway:right",
            "cycleway:both",
            "surface",
            "smoothness",
            "tracktype",
            "segregated",
            "lit",
            "lanes",
            "maxspeed",
            "oneway",
            "access",
            "foot",
            "horse",
            "motor_vehicle",
            "vehicle",
            "service",
            "junction",
            "bridge",
            "tunnel",
        ]
    )
)

# ============================================================
# LECTURE DEPARTEMENTS
# ============================================================

dep = gpd.read_file(DEP_FILE)

print(dep.columns)
print(dep.crs)

# ============================================================
# ADAPTER LES NOMS DE COLONNES ICI
# ============================================================

CODE_COL = "code_insee"      # ex : 67,68
NAME_COL = "nom_officiel"       # ex : Bas-Rhin

# ============================================================
# SELECTION BAS-RHIN / HAUT-RHIN
# ============================================================

dep67 = dep[dep[CODE_COL].astype(str) == "67"].copy()
dep68 = dep[dep[CODE_COL].astype(str) == "68"].copy()

deps = gpd.GeoDataFrame(
    pd.concat([dep67, dep68], ignore_index=True),
    crs=dep.crs
)

print("Départements trouvés :", len(deps))

# ============================================================
# CRS METRIQUE POUR BUFFER
# ============================================================

deps = deps.to_crs(2154)

# Buffer 200 m
deps["geometry"] = deps.geometry.buffer(200)

# réparation
deps["geometry"] = deps.geometry.apply(make_valid)

# retour WGS84
deps = deps.to_crs(4326)

# ============================================================
# COLONNES
# ============================================================

KEEP_COLS = [
    "osmid",
    "highway",
    "name",
    "ref",
    "surface",
    "smoothness",
    "tracktype",
    "bicycle",
    "foot",
    "horse",
    "motor_vehicle",
    "vehicle",
    "cycleway",
    "cycleway:left",
    "cycleway:right",
    "cycleway:both",
    "oneway",
    "lanes",
    "maxspeed",
    "lit",
    "segregated",
    "access",
    "bridge",
    "tunnel",
    "junction",
    "service",
    "geometry"
]

# ============================================================
# TELECHARGEMENT
# ============================================================

for _, row in deps.iterrows():

    code_dep = str(row[CODE_COL])
    nom_dep = str(row[NAME_COL])

    print("\n" + "="*70)
    print(f"{code_dep} - {nom_dep}")

    polygon = make_valid(row.geometry)

    if polygon.geom_type == "MultiPolygon":
        polygon = max(polygon.geoms, key=lambda g: g.area)

    print("Téléchargement OSM...")

    G = ox.graph_from_polygon(
        polygon,
        network_type="all",
        simplify=False,
        retain_all=True
    )

    edges = ox.graph_to_gdfs(
        G,
        nodes=False,
        fill_edge_geometry=True
    )

    cols = [c for c in KEEP_COLS if c in edges.columns]

    edges = edges[cols]

    outfile = os.path.join(
        OUTPUT_DIR,
        f"{code_dep}_{nom_dep.replace(' ','_')}.gpkg"
    )

    edges.to_file(
        outfile,
        layer="roads",
        driver="GPKG"
    )

    print(f"Routes : {len(edges):,}")
    print(outfile)

print("\nTERMINÉ")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Index(['cleabs', 'nom_officiel', 'nom_officiel_en_majuscules', 'code_insee',
       'code_insee_de_la_region', 'code_siren', 'nom', 'geometry'],
      dtype='object')
EPSG:3948
Départements trouvés : 2

67 - Bas-Rhin
Téléchargement OSM...
Routes : 2,440,936
/content/drive/MyDrive/data_osm_road/67_Bas-Rhin.gpkg

68 - Haut-Rhin
Téléchargement OSM...
Routes : 2,101,919
/content/drive/MyDrive/data_osm_road/68_Haut-Rhin.gpkg

TERMINÉ


# **Etape 2 : Contenu des fichiers**

In [ ]:
import geopandas as gpd
import pandas as pd


# ============================================================
# DRIVE
# ============================================================

from google.colab import drive
drive.mount('/content/drive')
# ============================================================

files = [
    "/content/drive/MyDrive/data_osm_road/67_Bas-Rhin.gpkg",
    "/content/drive/MyDrive/data_osm_road/68_Haut-Rhin.gpkg"
]

for file in files:
    print("\n" + "="*80)
    print(file)

    gdf = gpd.read_file(file)

    # Informations générales
    print("\n--- INFO GÉNÉRALES ---")
    print("Nombre de lignes :", len(gdf))
    print("Nombre de colonnes :", len(gdf.columns))

    print("\nColonnes :")
    for col in gdf.columns:
        print(f" - {col} ({gdf[col].dtype})")

    # Index
    print("\n--- INDEX ---")
    print(type(gdf.index))
    print(gdf.index)

    # Statistiques des colonnes
    print("\n--- VALEURS DISTINCTES PAR COLONNE ---")

    for col in gdf.columns:

        if col == "geometry":
            continue

        print(f"\n### {col}")

        nb_null = gdf[col].isna().sum()
        nb_unique = gdf[col].nunique(dropna=True)

        print(f"Valeurs uniques : {nb_unique}")
        print(f"Valeurs NULL : {nb_null}")

        vc = gdf[col].value_counts(dropna=False)

        # Limiter l'affichage si énorme
        if len(vc) > 100:
            print("Top 100 valeurs :")
            print(vc.head(100))
        else:
            print(vc)

Mounted at /content/drive

/content/drive/MyDrive/data_osm_road/67_Bas-Rhin.gpkg

--- INFO GÉNÉRALES ---
Nombre de lignes : 2440936
Nombre de colonnes : 30

Colonnes :
 - u (int64)
 - v (int64)
 - key (int64)
 - osmid (int64)
 - highway (object)
 - name (object)
 - ref (object)
 - surface (object)
 - smoothness (object)
 - tracktype (object)
 - bicycle (object)
 - foot (object)
 - horse (object)
 - motor_vehicle (object)
 - vehicle (object)
 - cycleway (object)
 - cycleway:left (object)
 - cycleway:right (object)
 - cycleway:both (object)
 - oneway (bool)
 - lanes (object)
 - maxspeed (object)
 - lit (object)
 - segregated (object)
 - access (object)
 - bridge (object)
 - tunnel (object)
 - junction (object)
 - service (object)
 - geometry (geometry)

--- INDEX ---
<class 'pandas.core.indexes.range.RangeIndex'>
RangeIndex(start=0, stop=2440936, step=1)

--- VALEURS DISTINCTES PAR COLONNE ---

### u
Valeurs uniques : 1194246
Valeurs NULL : 0
Top 100 valeurs :
u
2152402606    11
39149534

# **Etape 3 : Nettoyage des données**

## **Projection 3948 & tri attributaire**

In [ ]:
# ============================================================
# Reprojection et tri des attributs OSM (Google Colab)
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import geopandas as gpd

# ------------------------------------------------------------------
# Fichiers d'entrée
# ------------------------------------------------------------------

files = [
    "/content/drive/MyDrive/data_osm_road/67_Bas-Rhin.gpkg",
    "/content/drive/MyDrive/data_osm_road/68_Haut-Rhin.gpkg"
]

# ------------------------------------------------------------------
# Colonnes à conserver
# ------------------------------------------------------------------

cols_to_keep = [
    "osmid",
    "highway",
    "name",
    "ref",
    "bicycle",
    "cycleway",
    "cycleway:left",
    "cycleway:right",
    "cycleway:both",
    "oneway",
    "lanes",
    "maxspeed",
    "segregated",
    "access",
    "geometry"
]

# ------------------------------------------------------------------
# Traitement
# ------------------------------------------------------------------

for file in files:

    print("=" * 80)
    print(f"Traitement : {os.path.basename(file)}")

    # Lecture
    gdf = gpd.read_file(file)

    print(f"Projection initiale : {gdf.crs}")

    # --------------------------------------------------------------
    # (1) Reprojection EPSG:3948
    # --------------------------------------------------------------

    gdf_3948 = gdf.to_crs(epsg=3948)

    out_3948 = file.replace(".gpkg", "_3948.gpkg")

    gdf_3948.to_file(out_3948, driver="GPKG")

    print(f"✓ Sauvegardé : {out_3948}")

    # --------------------------------------------------------------
    # (2) Tri des attributs
    # --------------------------------------------------------------

    existing_cols = [c for c in cols_to_keep if c in gdf_3948.columns]

    gdf_tri = gdf_3948[existing_cols].copy()

    out_tri = file.replace(".gpkg", "_3948_tri_attributs.gpkg")

    gdf_tri.to_file(out_tri, driver="GPKG")

    print(f"✓ Sauvegardé : {out_tri}")

    # --------------------------------------------------------------
    # Résumé
    # --------------------------------------------------------------

    print(f"Nombre de lignes : {len(gdf_tri):,}")
    print(f"Nombre de colonnes conservées : {len(gdf_tri.columns)}")
    print("Colonnes :")
    for c in gdf_tri.columns:
        print(f" - {c}")

print("\nTraitement terminé.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Traitement : 67_Bas-Rhin.gpkg
Projection initiale : EPSG:4326
✓ Sauvegardé : /content/drive/MyDrive/data_osm_road/67_Bas-Rhin_3948.gpkg
✓ Sauvegardé : /content/drive/MyDrive/data_osm_road/67_Bas-Rhin_3948_tri_attributs.gpkg
Nombre de lignes : 2,440,936
Nombre de colonnes conservées : 15
Colonnes :
 - osmid
 - highway
 - name
 - ref
 - bicycle
 - cycleway
 - cycleway:left
 - cycleway:right
 - cycleway:both
 - oneway
 - lanes
 - maxspeed
 - segregated
 - access
 - geometry
Traitement : 68_Haut-Rhin.gpkg
Projection initiale : EPSG:4326
✓ Sauvegardé : /content/drive/MyDrive/data_osm_road/68_Haut-Rhin_3948.gpkg
✓ Sauvegardé : /content/drive/MyDrive/data_osm_road/68_Haut-Rhin_3948_tri_attributs.gpkg
Nombre de lignes : 2,101,919
Nombre de colonnes conservées : 15
Colonnes :
 - osmid
 - highway
 - name
 - ref
 - bicycle
 - cycleway
 - cycleway:left
 - cycleway:right


## **Tri segments (GRASSE v.clean sur Qgis)**

# **Etape 4 : Ajoute d'informations complémentaires _no**

In [ ]:
# ============================================================
# ETAPE 4
# Ajout d'informations complémentaires
#
# 4-1 : Traitement du champ REF
#   ref_al      = lettres uniquement
#   ref_ch      = chiffres uniquement
#   ref_concat  = concaténé (A4, D263...)
#
# 4-2 : Croisement zone agglomérée
#   AUT_GV = motorway / trunk
#   A      = dans zone agglo
#   HA     = hors zone agglo
#
# Entrées :
#   67_Bas-Rhin_3948_tri_attributs.gpkg
#   68_Haut-Rhin_3948_tri_attributs.gpkg
#
#   zone_agglo_bdtopo_2605.gpkg
#
# Sorties :
#   67_Bas-Rhin_3948_infos_1.gpkg
#   68_Haut-Rhin_3948_infos_2.gpkg
# ============================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import re
import numpy as np
import pandas as pd
import geopandas as gpd

# ============================================================
# PARAMETRES
# ============================================================

FILES = [
    "/content/drive/MyDrive/data_osm_road/67_Bas-Rhin_3948_tri_attributs.gpkg",
    "/content/drive/MyDrive/data_osm_road/68_Haut-Rhin_3948_tri_attributs.gpkg"
]

AGGLO_FILE = (
    "/content/drive/MyDrive/data_osm_road/"
    "zone_agglo_bdtopo_2605.gpkg"
)

# ============================================================
# LECTURE AGGLO
# ============================================================

print("=" * 80)
print("LECTURE ZONES AGGLO")

agglo = gpd.read_file(AGGLO_FILE)

print("CRS agglo :", agglo.crs)

if agglo.crs.to_epsg() != 3948:
    agglo = agglo.to_crs(3948)

print("CRS utilisé :", agglo.crs)
print("Nb polygones :", len(agglo))

# Dissolve pour accélérer le test spatial
agglo_union = agglo.dissolve()

print("Dissolve terminé")

# ============================================================
# TRAITEMENT
# ============================================================

for i, file in enumerate(FILES, start=1):

    print("\n")
    print("=" * 80)
    print(os.path.basename(file))

    # --------------------------------------------------------
    # Lecture
    # --------------------------------------------------------

    gdf = gpd.read_file(file)

    print("Lignes :", f"{len(gdf):,}")
    print("Colonnes :", len(gdf.columns))

    # --------------------------------------------------------
    # Vérification projection
    # --------------------------------------------------------

    if gdf.crs.to_epsg() != 3948:
        gdf = gdf.to_crs(3948)

    # --------------------------------------------------------
    # 4-1 REF
    # --------------------------------------------------------

    print("\nCréation ref_al / ref_ch / ref_concat")

    gdf["ref"] = gdf["ref"].fillna("").astype(str)

    # lettres
    gdf["ref_al"] = (
        gdf["ref"]
        .str.extract(r"([A-Za-z]+)", expand=False)
        .fillna("")
        .str.upper()
    )

    # chiffres
    gdf["ref_ch"] = (
        gdf["ref"]
        .str.extract(r"(\d+)", expand=False)
        .fillna("")
    )

    # concat
    gdf["ref_concat"] = (
        gdf["ref_al"].astype(str)
        + gdf["ref_ch"].astype(str)
    )

    gdf.loc[
        gdf["ref_concat"].str.strip() == "",
        "ref_concat"
    ] = None

    # --------------------------------------------------------
    # 4-2 AGGLO
    # --------------------------------------------------------

    print("\nCroisement spatial avec zones agglomérées")

    gdf["agglo"] = "HA"

    # autoroutes et voies rapides
    gv_mask = gdf["highway"].isin([
        "motorway",
        "motorway_link",
        "trunk",
        "trunk_link"
    ])

    gdf.loc[gv_mask, "agglo"] = "AUT_GV"

    # uniquement les autres
    test = gdf.loc[~gv_mask].copy()

    print("Segments à tester :", f"{len(test):,}")

    # centroïdes
    test["geometry"] = test.geometry.centroid

    # jointure spatiale
    join = gpd.sjoin(
        test,
        agglo_union,
        predicate="within",
        how="left"
    )

    inside_ids = join.loc[
        join.index_right.notna()
    ].index.unique()

    gdf.loc[inside_ids, "agglo"] = "A"

    # --------------------------------------------------------
    # STATISTIQUES
    # --------------------------------------------------------

    print("\nRépartition agglo :")

    print(
        gdf["agglo"]
        .value_counts(dropna=False)
    )

    # --------------------------------------------------------
    # EXPORT
    # --------------------------------------------------------

    dep_name = os.path.basename(file)

    dep_name = (
        dep_name
        .replace("_3948_tri_attributs.gpkg", "")
    )

    out_file = (
        f"/content/drive/MyDrive/data_osm_road/"
        f"{dep_name}_3948_infos_{i}.gpkg"
    )

    gdf.to_file(
        out_file,
        driver="GPKG"
    )

    print("\nFichier exporté :")
    print(out_file)

    # --------------------------------------------------------
    # INDEX
    # --------------------------------------------------------

    print("\nINDEX DU FICHIER")

    print("Nombre lignes :", f"{len(gdf):,}")
    print("Nombre colonnes :", len(gdf.columns))

    print("\nColonnes :")

    for c in gdf.columns:
        print("- ", c)

# ============================================================
# FIN
# ============================================================

print("\n" + "=" * 80)
print("TRAITEMENT TERMINE")
print("=" * 80)

print("""
Méthode appliquée :

1. Lecture des réseaux OSM déjà reprojetés en EPSG:3948

2. Traitement du champ REF
   - ref_al     : lettres
   - ref_ch     : chiffres
   - ref_concat : concaténation

3. Croisement avec les zones agglomérées BD TOPO

4. Création du champ AGGLO
   - AUT_GV : motorway / trunk
   - A      : intérieur zone agglo
   - HA     : hors zone agglo

5. Export GeoPackage final
""")

Mounted at /content/drive
LECTURE ZONES AGGLO
CRS agglo : EPSG:3948
CRS utilisé : EPSG:3948
Nb polygones : 1119
Dissolve terminé


67_Bas-Rhin_3948_tri_attributs.gpkg
Lignes : 2,440,936
Colonnes : 15

Création ref_al / ref_ch / ref_concat

Croisement spatial avec zones agglomérées
Segments à tester : 2,423,284

Répartition agglo :
agglo
HA        1460822
A          962462
AUT_GV      17652
Name: count, dtype: int64

Fichier exporté :
/content/drive/MyDrive/data_osm_road/67_Bas-Rhin_3948_infos_1.gpkg

INDEX DU FICHIER
Nombre lignes : 2,440,936
Nombre colonnes : 19

Colonnes :
-  osmid
-  highway
-  name
-  ref
-  bicycle
-  cycleway
-  cycleway:left
-  cycleway:right
-  cycleway:both
-  oneway
-  lanes
-  maxspeed
-  segregated
-  access
-  geometry
-  ref_al
-  ref_ch
-  ref_concat
-  agglo


68_Haut-Rhin_3948_tri_attributs.gpkg
Lignes : 2,101,919
Colonnes : 15

Création ref_al / ref_ch / ref_concat

Croisement spatial avec zones agglomérées
Segments à tester : 2,092,622

Répartition a

# **Etape 5 : Ajoute infos cyclables _ no**

In [ ]:
# ============================================================
# ETAPE 5
# POTENTIEL CYCLABLE
#
# Entrées :
#   67_Bas-Rhin_3948_infos_1.gpkg
#   68_Haut-Rhin_3948_infos_2.gpkg
#
# Sorties :
#   67_Bas-Rhin_3948_infos_cyclable.gpkg
#   68_Haut-Rhin_3948_infos_cyclable.gpkg
#
# Nouveaux champs :
#   cycle_statut
#   cycle_classe
#
# ============================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import geopandas as gpd
import pandas as pd

# ============================================================
# FICHIERS
# ============================================================

FILES = [
    "/content/drive/MyDrive/data_osm_road/67_Bas-Rhin_3948_infos_1.gpkg",
    "/content/drive/MyDrive/data_osm_road/68_Haut-Rhin_3948_infos_2.gpkg"
]

# ============================================================
# CLASSIFICATION
# ============================================================

def classify_cycle(row):

    highway = str(row.get("highway", "")).lower()

    bicycle = str(
        row.get("bicycle", "")
    ).lower()

    cycleway = str(
        row.get("cycleway", "")
    ).lower()

    cycle_left = str(
        row.get("cycleway:left", "")
    ).lower()

    cycle_right = str(
        row.get("cycleway:right", "")
    ).lower()

    cycle_both = str(
        row.get("cycleway:both", "")
    ).lower()

    segregated = str(
        row.get("segregated", "")
    ).lower()

    # =====================================================
    # AUTOROUTE
    # =====================================================

    if highway in [
        "motorway",
        "motorway_link"
    ]:
        return ("autoroute", None)

    # =====================================================
    # INTERDICTION
    # =====================================================

    if bicycle in [
        "no",
        "private",
        "customers"
    ]:
        return ("interdiction", None)

    # =====================================================
    # AMENAGEMENT CYCLABLE EXISTANT
    # =====================================================

    amenagements = [

        "track",

        "lane",

        "shared_lane",

        "share_busway",

        "opposite_lane",

        "opposite_track"

    ]

    if (
        highway == "cycleway"
        or cycleway in amenagements
        or cycle_left in amenagements
        or cycle_right in amenagements
        or cycle_both in amenagements
        or segregated == "yes"
    ):
        return ("cyclable_ex_ame", None)

    # -----------------------------------------------------

    if (
        bicycle == "designated"
        and highway in [
            "path",
            "footway"
        ]
    ):
        return ("cyclable_ex_ame", None)

    # =====================================================
    # CYCLABLE SANS AMENAGEMENT
    # =====================================================

    if highway in [
        "living_street"
    ]:
        return ("cyclable_sans_ame", None)

    # =====================================================
    # POTENTIEL D'AMENAGEMENT
    # =====================================================

    # Classe 1

    if highway == "path":

        return (
            "potentiel",
            "classe1"
        )

    # Classe 2

    if highway in [
        "residential",
        "service"
    ]:

        return (
            "potentiel",
            "classe2"
        )

    # Classe 3

    if highway in [
        "unclassified",
        "track",
        "busway"
    ]:

        return (
            "potentiel",
            "classe3"
        )

    # Classe 4

    if highway in [
        "tertiary",
        "tertiary_link"
    ]:

        return (
            "potentiel",
            "classe4"
        )

    # Classe 5

    if highway in [
        "secondary",
        "secondary_link"
    ]:

        return (
            "potentiel",
            "classe5"
        )

    # Classe 6

    if highway in [
        "primary",
        "primary_link",
        "trunk",
        "trunk_link"
    ]:

        return (
            "potentiel",
            "classe6"
        )

    # =====================================================
    # AUTRES CAS
    # =====================================================

    return (
        "interdiction",
        None
    )

# ============================================================
# TRAITEMENT
# ============================================================

for file in FILES:

    print("\n")
    print("=" * 80)
    print(os.path.basename(file))

    # --------------------------------------------------------
    # Lecture
    # --------------------------------------------------------

    gdf = gpd.read_file(file)

    print(f"Lignes : {len(gdf):,}")
    print(f"Colonnes : {len(gdf.columns)}")

    # --------------------------------------------------------
    # Classification
    # --------------------------------------------------------

    print("\nClassification...")

    result = gdf.apply(
        classify_cycle,
        axis=1,
        result_type="expand"
    )

    gdf["cycle_statut"] = result[0]
    gdf["cycle_classe"] = result[1]

    # --------------------------------------------------------
    # Statistiques
    # --------------------------------------------------------

    print("\n")
    print("=" * 60)
    print("CYCLE_STATUT")
    print("=" * 60)

    print(
        gdf["cycle_statut"]
        .value_counts(dropna=False)
    )

    print("\n")
    print("=" * 60)
    print("CYCLE_CLASSE")
    print("=" * 60)

    print(
        gdf["cycle_classe"]
        .value_counts(dropna=False)
    )

    # --------------------------------------------------------
    # Export
    # --------------------------------------------------------

    out_file = (
        file
        .replace("_infos_1.gpkg", "_infos_cyclable.gpkg")
        .replace("_infos_2.gpkg", "_infos_cyclable.gpkg")
    )

    gdf.to_file(
        out_file,
        driver="GPKG"
    )

    print("\n✓ Export :")
    print(out_file)

    # --------------------------------------------------------
    # Résumé
    # --------------------------------------------------------

    print("\n")
    print("=" * 60)
    print("RESUME")
    print("=" * 60)

    print(f"Lignes : {len(gdf):,}")
    print(f"Colonnes : {len(gdf.columns)}")

    print("\nListe des colonnes :")

    for c in gdf.columns:
        print(" -", c)

# ============================================================
# FIN
# ============================================================

print("\n")
print("=" * 80)
print("TRAITEMENT TERMINE")
print("=" * 80)

print("""

Méthode appliquée
-----------------

1. Détection des autoroutes

   highway :
   - motorway
   - motorway_link

2. Détection des interdictions

   bicycle :
   - no
   - private
   - customers

3. Détection des aménagements cyclables existants

   highway :
   - cycleway

   cycleway* :
   - track
   - lane
   - shared_lane
   - share_busway
   - opposite_lane
   - opposite_track

   segregated :
   - yes

   bicycle=designated
   sur :
   - path
   - footway

4. Détection des voies cyclables sans aménagement

   highway :
   - living_street

5. Classement du potentiel d'aménagement

   classe1 :
   - path

   classe2 :
   - residential
   - service

   classe3 :
   - unclassified
   - track
   - busway

   classe4 :
   - tertiary
   - tertiary_link

   classe5 :
   - secondary
   - secondary_link

   classe6 :
   - primary
   - primary_link
   - trunk
   - trunk_link

6. Export GeoPackage

""")

Mounted at /content/drive


67_Bas-Rhin_3948_infos_1.gpkg
Lignes : 2,440,936
Colonnes : 19

Classification...


CYCLE_STATUT
cycle_statut
potentiel            2102652
interdiction          210339
cyclable_ex_ame       113727
autoroute               7995
cyclable_sans_ame       6223
Name: count, dtype: int64


CYCLE_CLASSE
cycle_classe
classe3    990977
classe2    637953
None       338284
classe1    211871
classe4    139919
classe5     90060
classe6     31872
Name: count, dtype: int64

✓ Export :
/content/drive/MyDrive/data_osm_road/67_Bas-Rhin_3948_infos_cyclable.gpkg


RESUME
Lignes : 2,440,936
Colonnes : 21

Liste des colonnes :
 - osmid
 - highway
 - name
 - ref
 - bicycle
 - cycleway
 - cycleway:left
 - cycleway:right
 - cycleway:both
 - oneway
 - lanes
 - maxspeed
 - segregated
 - access
 - ref_al
 - ref_ch
 - ref_concat
 - agglo
 - geometry
 - cycle_statut
 - cycle_classe


68_Haut-Rhin_3948_infos_2.gpkg
Lignes : 2,101,919
Colonnes : 19

Classification...


CYCLE_STATUT
cycle_sta

# **Etape 6 : Ajoute infos vitesse proxy **

In [ ]:
# ============================================================
# ETAPE 6
# AJOUT VITESSE PROXY
#
# Entrées :
#   67_Bas-Rhin_3948_infos_cyclable.gpkg
#   68_Haut-Rhin_3948_infos_cyclable.gpkg
#
# Sorties :
#   67_Bas-Rhin_3948_infos_vitesse.gpkg
#   68_Haut-Rhin_3948_infos_vitesse.gpkg
#
# Nouveau champ :
#   vitesse_proxy
#
# Convention :
#   30   = valeur OSM
#   30*  = valeur proxy
#   99*  = inconnu
#
# ============================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import re
import geopandas as gpd
import pandas as pd

# ============================================================
# FICHIERS
# ============================================================

FILES = [
    "/content/drive/MyDrive/data_osm_road/67_Bas-Rhin_3948_infos_cyclable.gpkg",
    "/content/drive/MyDrive/data_osm_road/68_Haut-Rhin_3948_infos_cyclable.gpkg"
]

# ============================================================
# FONCTION
# ============================================================

def vitesse_proxy(row):

    maxspeed = row.get("maxspeed")
    highway = str(row.get("highway", "")).lower()
    agglo = str(row.get("agglo", "")).upper()

    cycleway = str(row.get("cycleway", "")).lower()

    # --------------------------------------------------------
    # MAXSPEED OSM
    # --------------------------------------------------------

    if pd.notna(maxspeed):

        ms = str(maxspeed).strip()

        # FRANCE / SUISSE

        if ms == "FR:zone30":
            return "30"

        if ms == "FR:urban":
            return "50"

        if ms in ["FR:rural", "CH:rural"]:
            return "80"

        if ms == "FR:motorway":
            return "130"

        # Valeur numérique

        m = re.search(r"(\d+)", ms)

        if m:

            v = int(m.group(1))

            if 1 <= v <= 9:
                return "1-9"

            if 11 <= v <= 19:
                return "11-19"

            if 21 <= v <= 29:
                return "21-29"

            return str(v)

    # --------------------------------------------------------
    # PROXY 0*
    # --------------------------------------------------------

    if highway in [
        "steps",
        "bridleway",
        "elevator",
        "corridor",
        "path",
        "escape",
        "footway",
        "pedestrian",
        "bus_stop"
    ]:
        return "0*"

    if cycleway == "track":
        return "0*"

    # --------------------------------------------------------
    # PROXY 20*
    # --------------------------------------------------------

    if highway == "living_street":
        return "20*"

    # --------------------------------------------------------
    # PROXY 30*
    # --------------------------------------------------------

    if highway in [
        "residential",
        "service",
        "busway",
        "crossing",
        "track"
    ]:
        return "30*"

    # --------------------------------------------------------
    # PROXY 50*
    # --------------------------------------------------------

    if (
        agglo == "A"
        and highway in [
            "primary",
            "secondary",
            "tertiary",
            "primary_link",
            "secondary_link",
            "tertiary_link",
            "unclassified"
        ]
    ):
        return "50*"

    # --------------------------------------------------------
    # PROXY 90*
    # --------------------------------------------------------

    if (
        agglo == "HA"
        and highway in [
            "primary",
            "secondary",
            "tertiary",
            "primary_link",
            "secondary_link",
            "tertiary_link"
        ]
    ):
        return "90*"

    # --------------------------------------------------------
    # PROXY 120*
    # --------------------------------------------------------

    if highway in [
        "trunk",
        "trunk_link"
    ]:
        return "120*"

    # --------------------------------------------------------
    # PROXY 130*
    # --------------------------------------------------------

    if highway in [
        "motorway",
        "motorway_link"
    ]:
        return "130*"

    # --------------------------------------------------------
    # AUTRES CYCLEWAY
    # --------------------------------------------------------

    if highway == "cycleway":
        return "15*"

    # --------------------------------------------------------
    # INCONNU
    # --------------------------------------------------------

    return "99*"

# ============================================================
# TRAITEMENT
# ============================================================

for file in FILES:

    print("\n" + "="*80)
    print(os.path.basename(file))

    # --------------------------------------------------------

    gdf = gpd.read_file(file)

    print(f"Lignes : {len(gdf):,}")

    # --------------------------------------------------------

    print("Calcul vitesse_proxy...")

    gdf["vitesse_proxy"] = gdf.apply(
        vitesse_proxy,
        axis=1
    )

    # --------------------------------------------------------
    # STATISTIQUES
    # --------------------------------------------------------

    print("\nDistribution vitesse_proxy")

    print(
        gdf["vitesse_proxy"]
        .value_counts(dropna=False)
        .sort_index()
    )

    # --------------------------------------------------------
    # EXPORT
    # --------------------------------------------------------

    outfile = file.replace(
        "_infos_cyclable.gpkg",
        "_infos_vitesse.gpkg"
    )

    gdf.to_file(
        outfile,
        driver="GPKG"
    )

    print("\n✓ Export")
    print(outfile)

    # --------------------------------------------------------

    print("\nColonnes :")

    for c in gdf.columns:
        print("-", c)

# ============================================================
# FIN
# ============================================================

print("\n" + "="*80)
print("TRAITEMENT TERMINE")
print("="*80)

print("""

Méthode appliquée
-----------------

1. Utilisation de maxspeed OSM lorsque disponible

    1-9
    10
    11-19
    20
    21-29
    30
    35
    40
    45
    48
    50
    60
    70
    80
    90
    100
    110
    120
    130

2. Conversion des valeurs textuelles

    FR:zone30   -> 30
    FR:urban    -> 50
    FR:rural    -> 80
    CH:rural    -> 80
    FR:motorway -> 130

3. Attribution d'une vitesse proxy (*)

    0*
    15*
    20*
    30*
    50*
    90*
    120*
    130*

4. Cas non déterminables

    99*

""")

Mounted at /content/drive

67_Bas-Rhin_3948_infos_cyclable.gpkg
Lignes : 2,440,936
Calcul vitesse_proxy...

Distribution vitesse_proxy
vitesse_proxy
0*        446263
1-9           29
10          1628
100          442
11-19        123
110         3036
120*        3001
130         3143
130*        1456
15*        67924
20          9302
20*         2062
21-29         18
30         85216
30*      1447380
35           308
40         31362
45           133
48             2
50         68835
50*        85557
60           142
70          8230
80         42359
90          5748
90*        80213
99*        47024
Name: count, dtype: int64

✓ Export
/content/drive/MyDrive/data_osm_road/67_Bas-Rhin_3948_infos_vitesse.gpkg

Colonnes :
- osmid
- highway
- name
- ref
- bicycle
- cycleway
- cycleway:left
- cycleway:right
- cycleway:both
- oneway
- lanes
- maxspeed
- segregated
- access
- ref_al
- ref_ch
- ref_concat
- agglo
- cycle_statut
- cycle_classe
- geometry
- vitesse_proxy

68_Haut-Rhin_3948_infos